## Middleware

- These are basically hooks which can act as a trigger for a tool call or a function.
- Can be used to control what happens inside an agent.
- The hooks can be retries, fall backs, rate limits, tranforming prompts, tool selection and output formatting etc.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

There are few in-built hooks too like 
- Summarization
- Human In Feedback
... more in Langchain

### Summarization
automatically summarize when approaching token limits

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage


agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:groq/compound-mini",  # We can use flash or mini models for summarization to save tokens
            trigger=("messages", 10),  # At every 10 messages summarization will work.
            keep=("messages", 4),  # To have enough context keep last 4 chats
        )
    ],
)

### What is a thread?

A **thread** is one conversation, identified by a `thread_id`. It's the key the `checkpointer` stores state under.

- Pass the **same** `thread_id` → the checkpointer loads that conversation's saved messages, appends the new turn, and saves again. This is what makes the agent remember.
- Pass a **new** `thread_id` → fresh conversation, empty state. Nothing carries over.
- Pass **no** `config` → error, because the checkpointer has no key to save under.

The agent itself is stateless. Memory lives in the checkpointer, and `thread_id` is how you address it — so one agent can serve many independent conversations (e.g. one `thread_id` per user).

`InMemorySaver` keeps this in RAM, so it's gone when the kernel restarts. For persistence across runs, swap in a durable checkpointer such as `SqliteSaver` or `PostgresSaver`.

In [3]:
config = {"configurable": {"thread_id": "testing-01"}}

In [4]:
import re

from IPython.display import Markdown, display


def strip_thinking(text: str) -> str:
    """qwen wraps reasoning in <think>...</think>, which Markdown would swallow as HTML."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()


def find_summary(messages):
    """The middleware injects the summary as a HumanMessage tagged lc_source=summarization."""
    return next(
        (m.content for m in messages if m.additional_kwargs.get("lc_source") == "summarization"),
        None,
    )


questions_list = [
    "In one sentence, what is a vector database?",
    "Name two popular ones.",
    "Which one is easiest to run locally?",
    "How is it different from a normal SQL database?",
    "What is an embedding?",
    "Which model would you use to create embeddings?",
    "What does cosine similarity measure?",
    "Now, what was the very first thing I asked you about?",
]


last_summary = None

for q in questions_list:
    result = agent.invoke({"messages": [HumanMessage(content=q)]}, config)

    display(Markdown(f"**Q:** {q}"))
    display(Markdown(strip_thinking(result["messages"][-1].content)))

    # Show the summary only on the turn it changes, i.e. when summarization actually fired.
    summary = find_summary(result["messages"])
    if summary and summary != last_summary:
        display(Markdown("---\n### 🗜️ Context summarized — older messages replaced by:"))
        display(Markdown(strip_thinking(summary)))
        display(Markdown("---"))
        last_summary = summary

    print(f"Message count in state: {len(result['messages'])}\n")

**Q:** In one sentence, what is a vector database?

A vector database is a specialized storage system designed to index and query high-dimensional numerical embeddings, enabling rapid similarity searches that power modern AI applications like semantic search, recommendation engines, and large language model integrations.

Message count in state: 2



**Q:** Name two popular ones.

Two popular vector databases are **Pinecone** and **Weaviate**.

...

Cosine similarity measures **how closely two vectors point in the same direction**, ignoring their magnitude. It calculates the cosine of the angle between them, producing a score between **-1 and 1**:

- **1** → identical direction (maximally similar)
- **0** → orthogonal (no meaningful similarity)
- **-1** → opposite direction (maximally dissimilar)

**Why it matters for embeddings:**  
Embedding models encode meaning in the *orientation* of vectors, not their length. Two texts of very different lengths might have vectors with different magnitudes, but if they convey similar concepts, their vectors will point in nearly the same direction → high cosine similarity. This makes it the default metric for semantic search, recommendation engines, and vector databases.

*(Mathematically: `cos(θ) = (A·B) / (||A|| × ||B||)`. Most modern text embeddings produce non-negative values, so scores typically fall between 0 and 1.)*

Want a quick Python snippet showing how to compute it with `numpy` or `faiss`, or how vector databases use it under the hood?

Message count in state: 8



**Q:** Now, what was the very first thing I asked you about?

Based on the conversation summary provided, your very first question asked me to **explain what a vector database is, provide examples, and compare how easy they are to deploy locally** (which led into the Pinecone vs. Weaviate discussion).

Message count in state: 10



In [5]:
# Inspect what is actually stored in this thread right now (no API call).
# After summarization, the early Q&A is gone and replaced by the summary message.
for m in agent.get_state(config).values["messages"]:
    tag = m.additional_kwargs.get("lc_source", "")
    print(f"{type(m).__name__:12} {tag:14} {strip_thinking(m.content)[:70]!r}")

HumanMessage summarization  'Here is a summary of the conversation to date:\n\n## SESSION INTENT\nExpl'
AIMessage                   'A SQL database is built for structured, relational data and excels at '
HumanMessage                'What is an embedding?'
AIMessage                   'An embedding is a numerical representation of data (like text, images,'
HumanMessage                'Which model would you use to create embeddings?'
AIMessage                   'The model you choose depends on your **data type**, but for **text** ('
HumanMessage                'What does cosine similarity measure?'
AIMessage                   'Cosine similarity measures **how closely two vectors point in the same'
HumanMessage                'Now, what was the very first thing I asked you about?'
AIMessage                   'Based on the conversation summary provided, your very first question a'
